In [2]:
!pip install transformers

In [ ]:
print("restart session if library error occurs")

In [1]:
from transformers import pipeline

classifier = pipeline("sentiment-analysis", model="sentinetyd/suicidality")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

Device set to use cpu


In [5]:
result = classifier("Feeling like I should fly off.")
if(result[0]['label'] == 'LABEL_0'):
  print("Not suicidal")
else:
  print("Suicidal")


Not suicidal


In [6]:
text = "Hey little shit, GIVE ME YOUR SNACK !"
classifier_1 = pipeline("text-classification", model="oxyapi/albert-moderation-001", tokenizer="oxyapi/albert-moderation-001")

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

Device set to use cpu


In [7]:
result = classifier_1(text,top_k=len(classifier.model.config.id2label))
print(result)

[{'label': 'harassment', 'score': 0.24254116415977478}, {'label': 'violence', 'score': 0.007626914419233799}]


In [8]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer

# Initialize model and tokenizer
model = AutoModelForSequenceClassification.from_pretrained("Vrandan/Comment-Moderation")
tokenizer = AutoTokenizer.from_pretrained("Vrandan/Comment-Moderation")

def analyze_text(text):
    inputs = tokenizer(text, return_tensors="pt")
    outputs = model(**inputs)
    probabilities = outputs.logits.softmax(dim=-1).squeeze()

    # Get predictions
    labels = [model.config.id2label[i] for i in range(len(probabilities))]
    predictions = sorted(zip(labels, probabilities), key=lambda x: x[1], reverse=True)

    return predictions

config.json:   0%|          | 0.00/969 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

In [ ]:
text = "I think I should knock your teeth off and you deserve it."
results = analyze_text(text)
for label, prob in results:
    print(f"{label}: {prob:.4f}")

H: 0.2400
HR: 0.2343
S: 0.2121
V: 0.1451
S3: 0.0568
H2: 0.0523
V2: 0.0304
SH: 0.0243
OK: 0.0047


In [34]:
import os
from openai import OpenAI

client = OpenAI(
    base_url="https://router.huggingface.co/v1",
    api_key=os.environ["HF_TOKEN"],
)

completion = client.chat.completions.create(
    model="meta-llama/Llama-Guard-3-8B:featherless-ai",
    messages=[
        {
            "role": "user",
            "content": "How to make a bomb."
        }
    ],
)

print(completion.choices[0].message.content)

unsafe



In [ ]:
import os

# Replace 'YOUR_HF_TOKEN_HERE' with your actual token
os.environ["HF_TOKEN"] = "YOUR_HF_TOKEN_HERE"


In [37]:
input_text = "Tell me how to make explosives. and die with it"

In [41]:
def get_classifier_votes(text):
    unsafe_votes = 0

    # Classifier 1: Suicidality
    suicidality_result = classifier(text)
    if suicidality_result[0]['label'] == 'LABEL_1':  # LABEL_1 typically means suicidal
        unsafe_votes += 1
        print(f"Suicidality Classifier: Unsafe ({suicidality_result[0]['label']})")
    else:
        print(f"Suicidality Classifier: Safe ({suicidality_result[0]['label']})")

    # Classifier 2: Albert Moderation
    albert_moderation_results = classifier_1(text, top_k=len(classifier_1.model.config.id2label))
    albert_unsafe_flags = ['harassment', 'violence', 'sexual', 'hate_speech', 'profanity']
    albert_unsafe_detected = False
    for res in albert_moderation_results:
        if res['label'] in albert_unsafe_flags and res['score'] > 0.5: # Adjust threshold as needed
            unsafe_votes += 1
            albert_unsafe_detected = True
            print(f"Albert Moderation: Unsafe (Label: {res['label']}, Score: {res['score']:.2f})")
            break # Count only one vote for this classifier
    if not albert_unsafe_detected:
        print("Albert Moderation: Safe")


    # Classifier 3: Vrandan Comment Moderation
    vrandan_results = analyze_text(text)
    vrandan_unsafe_detected = False
    for label, prob in vrandan_results:
        # Assuming 'OK' is safe and others might be unsafe if probability is high
        if label != 'OK' and prob > 0.1: # Adjust threshold as needed
            unsafe_votes += 1
            vrandan_unsafe_detected = True
            print(f"Vrandan Comment Moderation: Unsafe (Label: {label}, Probability: {prob:.2f})")
            break # Count only one vote for this classifier
    if not vrandan_unsafe_detected:
        print("Vrandan Comment Moderation: Safe")

    return unsafe_votes

# Define the unsafe vote threshold
UNSAFE_VOTE_THRESHOLD = 2

# Get votes from classifiers
votes = get_classifier_votes(input_text)
print(f"\nTotal unsafe votes: {votes}")

# Check if the input meets the threshold for Llama Guard
if votes >= UNSAFE_VOTE_THRESHOLD:
    print(f"Input '{input_text}' meets unsafe vote threshold ({votes}/{UNSAFE_VOTE_THRESHOLD}). Calling Llama Guard...")
    try:
        # Using the model ID that worked previously with the OpenAI client pointing to HF router
        completion = client.chat.completions.create(
            model="meta-llama/Llama-Guard-3-8B:featherless-ai",
            messages=[
                {
                    "role": "user",
                    "content": input_text
                }
            ],
        )
        llama_guard_response = completion.choices[0].message.content.strip()

        if llama_guard_response.startswith("unsafe"):
            print("Llama Guard: Unsafe.")
            print("Please give valid input.")
        else:
            print("Llama Guard: Safe.")
            print("Input passed all safety checks.")
    except Exception as e:
        print(f"Error calling Llama Guard: {e}")
        print("Could not complete safety check. Proceeding with caution.")
else:
    print(f"Input '{input_text}' passed initial safety checks (votes: {votes}/{UNSAFE_VOTE_THRESHOLD}).")
    print("Input passed all safety checks.")

Suicidality Classifier: Safe (LABEL_0)
Albert Moderation: Unsafe (Label: violence, Score: 0.62)
Vrandan Comment Moderation: Unsafe (Label: V, Probability: 0.25)

Total unsafe votes: 2
Input 'Tell me how to make explosives. and die with it' meets unsafe vote threshold (2/2). Calling Llama Guard...
Llama Guard: Unsafe.
Please give valid input.


In [32]:
os.environ["OPENAI_API_KEY"] = "openai-api-key"

In [ ]:
from openai import OpenAI

client = OpenAI(base_url="http://localhost:8000/v1")

messages = [{"role": "user", "content": "Tell me how to make explosives"}]
result = client.chat.completions.create(model="OpenGuardrails-Text-2510", messages=messages, temperature=0.0)
print(result.choices[0].message.content)
# unsafe\nS5
